# BirdCLEF 2026 — Bird Sound Classification
**Pipeline:** Data Exploration → Feature Extraction → Traditional ML (KNN / SVM / RF) → CNN → BiLSTM

> **Kaggle 使用说明**
> 1. 在右侧 *Data* 面板添加 `birdclef-2026` 数据集
> 2. 打开 *Settings → Accelerator* 选择 **GPU T4 x2**
> 3. 依次运行全部 Cell

## 0. 环境安装 & 路径配置

In [ ]:
# Kaggle 已预装 librosa / torch，这里只补充 tqdm（一般也有）
import importlib, subprocess, sys
for pkg in ['tqdm', 'librosa', 'seaborn']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All packages ready.')

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR     = '/kaggle/input/competitions/birdclef-2026'
WORK_DIR     = '/kaggle/working'
FEATURES_DIR = os.path.join(WORK_DIR, 'features')
MODELS_DIR   = os.path.join(WORK_DIR, 'models')
RESULTS_DIR  = os.path.join(WORK_DIR, 'results')

for d in [FEATURES_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Audio ──────────────────────────────────────────────────────────────────
SR         = 22050
DURATION   = 5.0
N_SAMPLES  = int(SR * DURATION)  # 110250

# ── Feature extraction ────────────────────────────────────────────────────
N_MFCC    = 40
N_MELS    = 128
HOP_LENGTH = 512
N_FFT     = 2048
N_CHROMA  = 12
SPEC_FRAMES = 1 + (N_SAMPLES - N_FFT) // HOP_LENGTH  # ~212

# ── Dataset subset ────────────────────────────────────────────────────────
TOP_N_CLASSES  = 20
MAX_SAMPLES_ML = 50    # per class for traditional ML
MAX_SAMPLES_DL = 100   # per class for deep learning

# ── Training ──────────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.2
VAL_SIZE     = 0.1
BATCH_SIZE   = 32
EPOCHS_CNN   = 30
EPOCHS_LSTM  = 25
LR           = 1e-3
PATIENCE     = 7

print('Config loaded.')
print(f'DATA_DIR    = {DATA_DIR}')
print(f'WORK_DIR    = {WORK_DIR}')

In [ ]:
import os, glob

# ── 诊断：确认实际数据路径 ─────────────────────────────────────────────────
print('DATA_DIR exists:', os.path.exists(DATA_DIR))
print('Contents of DATA_DIR:')
for f in sorted(os.listdir(DATA_DIR)):
    print(' ', f)

# 看 train.csv 里 filename 列长什么样
import pandas as pd
df_check = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print('\nfilename column samples:')
print(df_check['filename'].head(5).tolist())

# 试着拼出第一个路径，看是否存在
first_fname = df_check['filename'].iloc[0]
candidate1 = os.path.join(DATA_DIR, 'train_audio', first_fname)
candidate2 = os.path.join(DATA_DIR, first_fname)
print(f'\ncandidate1: {candidate1}  exists={os.path.exists(candidate1)}')
print(f'candidate2: {candidate2}  exists={os.path.exists(candidate2)}')

## 1. 工具函数 (音频加载 / 特征提取)

In [ ]:
import numpy as np
import librosa
import warnings
warnings.filterwarnings('ignore')


def load_audio(filepath: str) -> np.ndarray:
    """Load audio, convert to mono, pad/trim to fixed length."""
    try:
        y, _ = librosa.load(filepath, sr=SR, mono=True, duration=DURATION)
    except Exception as e:
        raise RuntimeError(f'Cannot load {filepath}: {e}')
    target = N_SAMPLES
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y


def extract_features(y: np.ndarray) -> np.ndarray:
    """1-D statistical feature vector for traditional ML (278 dims)."""
    feats = []
    mfcc   = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    delta1 = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    for mat in (mfcc, delta1, delta2):
        feats.extend(mat.mean(axis=1))
        feats.extend(mat.std(axis=1))
    chroma = librosa.feature.chroma_stft(y=y, sr=SR, n_chroma=N_CHROMA, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.extend(chroma.mean(axis=1))
    feats.extend(chroma.std(axis=1))
    # spectral_centroid / bandwidth / rolloff accept sr; spectral_flatness does NOT
    for fn in (librosa.feature.spectral_centroid,
               librosa.feature.spectral_bandwidth,
               librosa.feature.spectral_rolloff):
        val = fn(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
        feats.append(float(val.mean()))
        feats.append(float(val.std()))
    flat = librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats.append(float(flat.mean()))
    feats.append(float(flat.std()))
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP_LENGTH)
    feats.append(float(zcr.mean())); feats.append(float(zcr.std()))
    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH)
    feats.append(float(rms.mean())); feats.append(float(rms.std()))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        f0 = librosa.yin(y, fmin=32, fmax=2048, sr=SR, hop_length=HOP_LENGTH)
    f0v = f0[f0 > 0]
    feats.append(float(f0v.mean()) if len(f0v) > 0 else 0.0)
    feats.append(float(f0v.std())  if len(f0v) > 0 else 0.0)
    return np.array(feats, dtype=np.float32)


def get_mel_spectrogram(y: np.ndarray) -> np.ndarray:
    """Log-mel spectrogram, shape (1, N_MELS, SPEC_FRAMES)."""
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    if log_mel.shape[1] < SPEC_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, SPEC_FRAMES - log_mel.shape[1])),
                         mode='constant', constant_values=log_mel.min())
    else:
        log_mel = log_mel[:, :SPEC_FRAMES]
    lo, hi = log_mel.min(), log_mel.max()
    if hi > lo:
        log_mel = (log_mel - lo) / (hi - lo)
    return log_mel[np.newaxis, :, :].astype(np.float32)


def get_mfcc_sequence(y: np.ndarray) -> np.ndarray:
    """MFCC time-series, shape (SPEC_FRAMES, N_MFCC)."""
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    if mfcc.shape[1] < SPEC_FRAMES:
        mfcc = np.pad(mfcc, ((0, 0), (0, SPEC_FRAMES - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :SPEC_FRAMES]
    mean = mfcc.mean(axis=1, keepdims=True)
    std  = mfcc.std(axis=1,  keepdims=True) + 1e-8
    mfcc = (mfcc - mean) / std
    return mfcc.T.astype(np.float32)


print('Utility functions defined.')

## 2. 数据探索

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa.display

df   = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
taxa = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))

print(f'Total samples  : {len(df)}')
print(f'Class breakdown:')
print(df['class_name'].value_counts())

birds = df[df['class_name'] == 'Aves'].copy()
print(f'\nBird (Aves) samples   : {len(birds)}')
print(f'Unique bird species   : {birds["primary_label"].nunique()}')

top_species = birds['primary_label'].value_counts().head(TOP_N_CLASSES)
print(f'\nTop {TOP_N_CLASSES} species:\n{top_species}')

In [ ]:
# Figure 1: Class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
top_species.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Number of samples')
axes[0].set_title(f'Top {TOP_N_CLASSES} Bird Species by Sample Count')
axes[0].invert_yaxis()

top_df = top_species.reset_index()
top_df.columns = ['primary_label', 'count']
top_df = top_df.merge(taxa[['primary_label', 'common_name']], on='primary_label', how='left')
top_df['label'] = top_df['common_name'].fillna(top_df['primary_label'])
top_df.set_index('label')['count'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Number of samples')
axes[1].set_title('Top Species with Common Names')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '01_class_distribution.png'), dpi=150)
plt.show()

In [ ]:
# Figure 2: Taxonomy pie
fig, ax = plt.subplots(figsize=(6, 6))
df['class_name'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%',
                                      startangle=140, colors=plt.cm.Set3.colors)
ax.set_ylabel('')
ax.set_title('Sample Distribution by Taxonomic Class')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '02_taxonomy_pie.png'), dpi=150)
plt.show()

In [ ]:
# Figure 3: Waveform / Mel / MFCC / Chroma for one sample
name_map = taxa.set_index('primary_label')['common_name'].to_dict()
sample_row = birds[birds['primary_label'] == top_species.index[0]].iloc[0]

_ar = globals().get('AUDIO_ROOT', os.path.join(DATA_DIR, 'train_audio'))
audio_path = os.path.join(_ar, sample_row['filename'])
if not os.path.exists(audio_path):
    audio_path = os.path.join(DATA_DIR, sample_row['filename'])
print(f'Sample: {audio_path}  exists={os.path.exists(audio_path)}')

y_s, sr_s = librosa.load(audio_path, sr=SR, mono=True, duration=DURATION)
if len(y_s) < N_SAMPLES:
    y_s = np.pad(y_s, (0, N_SAMPLES - len(y_s)))

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 2, figure=fig)

# ── Waveform ──────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
librosa.display.waveshow(y_s, sr=SR, ax=ax1, color='steelblue')
ax1.set_title(f'Waveform — {name_map.get(sample_row["primary_label"], sample_row["primary_label"])}')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')

# ── Mel-Spectrogram ───────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
mel_s     = librosa.feature.melspectrogram(y=y_s, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
log_mel_s = librosa.power_to_db(mel_s, ref=np.max, top_db=80)
img = librosa.display.specshow(log_mel_s, sr=SR, hop_length=HOP_LENGTH,
                                x_axis='time', y_axis='mel', ax=ax2)
fig.colorbar(img, ax=ax2, format='%+2.0f dB')
ax2.set_title('Mel-Spectrogram (dB, ref=max)')

# ── STFT Spectrogram ──────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
D = librosa.amplitude_to_db(np.abs(librosa.stft(y_s, n_fft=N_FFT, hop_length=HOP_LENGTH)), ref=np.max)
img2 = librosa.display.specshow(D, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', y_axis='log', ax=ax3)
fig.colorbar(img2, ax=ax3, format='%+2.0f dB')
ax3.set_title('STFT Spectrogram (log-freq axis, dB)')

# ── MFCC：对每个系数做 z-score（与喂给 LSTM 的预处理一致）──────────────
# C0 能量项绝对值远大于 C1-C39，不归一化则全图显示为白色
ax4 = fig.add_subplot(gs[2, 0])
mfcc_s = librosa.feature.mfcc(y=y_s, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
mfcc_norm = (mfcc_s - mfcc_s.mean(axis=1, keepdims=True)) / (mfcc_s.std(axis=1, keepdims=True) + 1e-8)
img3 = librosa.display.specshow(mfcc_norm, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', ax=ax4,
                                  cmap='RdBu_r', vmin=-3, vmax=3)
fig.colorbar(img3, ax=ax4, label='z-score')
ax4.set_title(f'MFCC ({N_MFCC} coefficients, per-coeff z-score)')
ax4.set_ylabel('MFCC index')

# ── Chroma ────────────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
chroma_s = librosa.feature.chroma_stft(y=y_s, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
img4 = librosa.display.specshow(chroma_s, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', y_axis='chroma', ax=ax5)
fig.colorbar(img4, ax=ax5)
ax5.set_title('Chroma Features')

plt.suptitle(f'Audio Analysis: {name_map.get(sample_row["primary_label"], sample_row["primary_label"])}',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '03_audio_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Mel + MFCC comparison across 4 species
# Mel:  ref=1.0 → 统一绝对 dB 基准，跨物种颜色可比
# MFCC: per-coeff z-score + RdBu_r → 消除 C0 能量项的主导，让 C1-C39 可见
fig, axes = plt.subplots(4, 2, figsize=(14, 14))
sample_labels = top_species.index[:4].tolist()
yy_last, title_last = None, ''

_ar4 = globals().get('AUDIO_ROOT', os.path.join(DATA_DIR, 'train_audio'))

for i, label in enumerate(sample_labels):
    row_i  = birds[birds['primary_label'] == label].iloc[0]
    path_i = os.path.join(_ar4, row_i['filename'])
    if not os.path.exists(path_i):
        path_i = os.path.join(DATA_DIR, row_i['filename'])
    try:
        yy, _ = librosa.load(path_i, sr=SR, mono=True, duration=DURATION)
        if len(yy) < N_SAMPLES:
            yy = np.pad(yy, (0, N_SAMPLES - len(yy)))

        mel_i  = librosa.power_to_db(
            librosa.feature.melspectrogram(y=yy, sr=SR, n_mels=N_MELS,
                                            n_fft=N_FFT, hop_length=HOP_LENGTH),
            ref=1.0, top_db=80)
        mfcc_i = librosa.feature.mfcc(y=yy, sr=SR, n_mfcc=N_MFCC,
                                       n_fft=N_FFT, hop_length=HOP_LENGTH)
        # per-coefficient z-score：与 LSTM 输入预处理一致，消除 C0 主导问题
        mfcc_norm = (mfcc_i - mfcc_i.mean(axis=1, keepdims=True)) / \
                    (mfcc_i.std(axis=1, keepdims=True) + 1e-8)

        common  = taxa[taxa['primary_label'] == label]['common_name'].values
        title_i = common[0] if len(common) > 0 else label
        yy_last, title_last = yy, title_i

        im0 = librosa.display.specshow(mel_i, sr=SR, hop_length=HOP_LENGTH,
                                        x_axis='time', y_axis='mel', ax=axes[i][0])
        fig.colorbar(im0, ax=axes[i][0], format='%+2.0f dB')
        axes[i][0].set_title(f'Mel-Spec (ref=1.0): {title_i}', fontsize=9)

        im1 = librosa.display.specshow(mfcc_norm, sr=SR, hop_length=HOP_LENGTH,
                                        x_axis='time', ax=axes[i][1],
                                        cmap='RdBu_r', vmin=-3, vmax=3)
        fig.colorbar(im1, ax=axes[i][1], label='z-score')
        axes[i][1].set_title(f'MFCC (z-score): {title_i}', fontsize=9)
        axes[i][1].set_ylabel('MFCC index')
    except Exception as e:
        print(f'  Skipped {label}: {e}')

plt.suptitle('Mel-Spectrogram and MFCC Comparison Across Species\n'
             '(Mel: absolute dB ref=1.0 for cross-species comparison; '
             'MFCC: per-coeff z-score, RdBu_r)',
             y=1.01, fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_species_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Pitch / RMS / ZCR for last loaded species
if yy_last is not None:
    f0  = librosa.yin(yy_last, fmin=32, fmax=2048, sr=SR, hop_length=HOP_LENGTH)
    rms = librosa.feature.rms(y=yy_last, hop_length=HOP_LENGTH)[0]
    zcr = librosa.feature.zero_crossing_rate(yy_last, hop_length=HOP_LENGTH)[0]
    t_f0  = librosa.times_like(f0,  sr=SR, hop_length=HOP_LENGTH)
    t_rms = librosa.times_like(rms, sr=SR, hop_length=HOP_LENGTH)

    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    axes[0].plot(t_f0,  f0,  color='green',      linewidth=0.8)
    axes[0].set_ylabel('Frequency (Hz)'); axes[0].set_title(f'Pitch (F0) — {title_last}'); axes[0].set_xlim(0, DURATION)
    axes[1].plot(t_rms, rms, color='darkorange',  linewidth=0.8)
    axes[1].set_ylabel('RMS Energy');    axes[1].set_title('Volume (RMS Energy)');           axes[1].set_xlim(0, DURATION)
    axes[2].plot(t_rms[:len(zcr)], zcr, color='purple', linewidth=0.8)
    axes[2].set_ylabel('Rate'); axes[2].set_xlabel('Time (s)'); axes[2].set_title('Zero Crossing Rate'); axes[2].set_xlim(0, DURATION)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, '05_pitch_energy_zcr.png'), dpi=150)
    plt.show()

print('Data exploration complete.')

## 3. 特征提取
提取三种特征并保存到 `/kaggle/working/features/`。
> 约 2000 个样本，T4 GPU 上约 **5~10 分钟**。

In [ ]:
import pickle, json
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

# ── 自动检测正确的音频根路径 ──────────────────────────────────────────────
df_tmp = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
_sample_fname = df_tmp['filename'].iloc[0]
if os.path.exists(os.path.join(DATA_DIR, 'train_audio', _sample_fname)):
    AUDIO_ROOT = os.path.join(DATA_DIR, 'train_audio')
elif os.path.exists(os.path.join(DATA_DIR, _sample_fname)):
    AUDIO_ROOT = DATA_DIR
else:
    # 搜索实际存在 .ogg 的目录
    _found = glob.glob(os.path.join(DATA_DIR, '**', '*.ogg'), recursive=True)
    if _found:
        # 推断 AUDIO_ROOT：去掉 filename 对应的后缀部分
        _fname_parts = _sample_fname.replace('\\', '/').split('/')
        _test_root = os.path.dirname(_found[0])
        for _ in range(len(_fname_parts) - 1):
            _test_root = os.path.dirname(_test_root)
        AUDIO_ROOT = _test_root
        print(f'Auto-detected AUDIO_ROOT: {AUDIO_ROOT}')
    else:
        raise FileNotFoundError(f'Cannot find any .ogg files under {DATA_DIR}')
print(f'AUDIO_ROOT = {AUDIO_ROOT}')
print(f'Sample path: {os.path.join(AUDIO_ROOT, _sample_fname)}  '
      f'exists={os.path.exists(os.path.join(AUDIO_ROOT, _sample_fname))}')

birds_all   = df[df['class_name'] == 'Aves'].copy()
top_sp_list = birds_all['primary_label'].value_counts().head(TOP_N_CLASSES).index.tolist()
birds_sub   = birds_all[birds_all['primary_label'].isin(top_sp_list)].copy()
print(f'Using {len(top_sp_list)} species, {len(birds_sub)} total samples before capping')

rng = np.random.default_rng(RANDOM_STATE)
selected_rows = []
for sp in top_sp_list:
    sp_rows = birds_sub[birds_sub['primary_label'] == sp]
    n   = min(len(sp_rows), MAX_SAMPLES_DL)
    idx = rng.choice(len(sp_rows), size=n, replace=False)
    selected_rows.append(sp_rows.iloc[idx])
selected = pd.concat(selected_rows).reset_index(drop=True)
print(f'Selected {len(selected)} samples')

le = LabelEncoder()
le.fit(top_sp_list)
with open(os.path.join(FEATURES_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

class_info = {sp: name_map.get(sp, sp) for sp in top_sp_list}
with open(os.path.join(FEATURES_DIR, 'class_info.json'), 'w') as f:
    json.dump(class_info, f, indent=2)

X_ml, X_mel, X_mfcc, y_all = [], [], [], []
failed = 0
MAX_ERR_PRINT = 3  # 最多打印前3个错误

for _, row in tqdm(selected.iterrows(), total=len(selected), desc='Extracting features'):
    fpath = os.path.join(AUDIO_ROOT, row['filename'])
    label = le.transform([row['primary_label']])[0]
    try:
        audio = load_audio(fpath)
        X_ml.append(extract_features(audio))
        X_mel.append(get_mel_spectrogram(audio))
        X_mfcc.append(get_mfcc_sequence(audio))
        y_all.append(label)
    except Exception as e:
        if failed < MAX_ERR_PRINT:
            print(f'\n[ERROR #{failed+1}] {fpath}\n  {e}')
        failed += 1

print(f'\nExtracted {len(y_all)} samples ({failed} failed)')

X_ml   = np.array(X_ml,   dtype=np.float32)
X_mel  = np.array(X_mel,  dtype=np.float32)
X_mfcc = np.array(X_mfcc, dtype=np.float32)
y_all  = np.array(y_all,  dtype=np.int64)

print(f'X_ml   shape: {X_ml.shape}')
print(f'X_mel  shape: {X_mel.shape}')
print(f'X_mfcc shape: {X_mfcc.shape}')

np.save(os.path.join(FEATURES_DIR, 'X_ml.npy'),        X_ml)
np.save(os.path.join(FEATURES_DIR, 'X_mel.npy'),       X_mel)
np.save(os.path.join(FEATURES_DIR, 'X_mfcc_seq.npy'),  X_mfcc)
np.save(os.path.join(FEATURES_DIR, 'y.npy'),           y_all)
np.save(os.path.join(FEATURES_DIR, 'label_names.npy'), np.array(le.classes_, dtype=str))
print('All features saved.')

## 4. 传统机器学习 (KNN / SVM / Random Forest)

In [ ]:
from collections import Counter
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import seaborn as sns

# ── Load features ─────────────────────────────────────────────────────────
X_m    = np.load(os.path.join(FEATURES_DIR, 'X_ml.npy'))
y_m    = np.load(os.path.join(FEATURES_DIR, 'y.npy'))
lnames = np.load(os.path.join(FEATURES_DIR, 'label_names.npy'))
with open(os.path.join(FEATURES_DIR, 'class_info.json')) as f:
    ci = json.load(f)

# Cap per-class for speed
counts = Counter(y_m)
keep_idx = []
rng2 = np.random.default_rng(RANDOM_STATE)
for cls, cnt in counts.items():
    idx = np.where(y_m == cls)[0]
    keep_idx.extend(rng2.choice(idx, size=min(cnt, MAX_SAMPLES_ML), replace=False).tolist())
keep_idx = sorted(keep_idx)
X_m = X_m[keep_idx]; y_m = y_m[keep_idx]
print(f'Using {len(y_m)} samples for traditional ML')

X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=TEST_SIZE,
                                            stratify=y_m, random_state=RANDOM_STATE)
short_names = [ci.get(sp, sp)[:12] for sp in lnames]
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def evaluate_and_save(name, model, Xtr, ytr, Xte, yte):
    print(f'\n{"="*50}\nModel: {name}')
    cv_sc = cross_val_score(model, Xtr, ytr, cv=cv5, scoring='accuracy', n_jobs=-1)
    print(f'  5-fold CV: {cv_sc.mean():.4f} ± {cv_sc.std():.4f}')
    model.fit(Xtr, ytr)
    yp  = model.predict(Xte)
    acc = accuracy_score(yte, yp)
    f1  = f1_score(yte, yp, average='macro')
    print(f'  Test acc: {acc:.4f}   Macro F1: {f1:.4f}')
    print(classification_report(yte, yp, target_names=lnames, zero_division=0))
    cm = confusion_matrix(yte, yp)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names, ax=ax, linewidths=0.3)
    ax.set(xlabel='Predicted', ylabel='True',
           title=f'{name} — Confusion Matrix\nAcc={acc:.3f}  F1={f1:.3f}')
    plt.xticks(rotation=45, ha='right', fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout()
    fname = f'cm_{name.lower().replace(" ", "_")}.png'
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150)
    plt.show()
    with open(os.path.join(MODELS_DIR, f'{name.lower().replace(" ", "_")}.pkl'), 'wb') as f:
        pickle.dump(model, f)
    return {'model': name, 'cv_mean': cv_sc.mean(), 'cv_std': cv_sc.std(),
            'test_acc': acc, 'macro_f1': f1}


# KNN
knn_pipe = Pipeline([('scaler', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1))])
gs_knn = GridSearchCV(knn_pipe, {'knn__n_neighbors': [3, 5, 7, 11, 15]},
                      cv=cv5, scoring='accuracy', n_jobs=-1)
gs_knn.fit(X_tr, y_tr)
print(f'KNN best k={gs_knn.best_params_["knn__n_neighbors"]}, CV={gs_knn.best_score_:.4f}')
r_knn = evaluate_and_save('KNN', gs_knn.best_estimator_, X_tr, y_tr, X_te, y_te)

# SVM
svm_pipe = Pipeline([('scaler', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=10, gamma='scale',
                                 decision_function_shape='ovr', random_state=RANDOM_STATE))])
r_svm = evaluate_and_save('SVM', svm_pipe, X_tr, y_tr, X_te, y_te)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
r_rf = evaluate_and_save('Random Forest', rf, X_tr, y_tr, X_te, y_te)

In [ ]:
# ML summary bar chart
results = [r_knn, r_svm, r_rf]
res_df  = pd.DataFrame(results).set_index('model')
print(res_df.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(res_df)); w = 0.3
ax.bar(x - w, res_df['cv_mean'],  w, label='CV Accuracy', color='steelblue',
       yerr=res_df['cv_std'], capsize=4)
ax.bar(x,     res_df['test_acc'], w, label='Test Accuracy', color='coral')
ax.bar(x + w, res_df['macro_f1'], w, label='Macro F1',      color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(res_df.index)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Traditional ML Model Comparison'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ml_comparison.png'), dpi=150)
plt.show()
res_df.to_csv(os.path.join(RESULTS_DIR, 'ml_results.csv'))

# Random Forest feature importance
importances = rf.feature_importances_
top_k = 20
top_idx = np.argsort(importances)[-top_k:][::-1]
feat_names = (
    [f'MFCC_{i}_mean' for i in range(40)] + [f'MFCC_{i}_std' for i in range(40)] +
    [f'dMFCC_{i}_mean' for i in range(40)] + [f'dMFCC_{i}_std' for i in range(40)] +
    [f'ddMFCC_{i}_mean' for i in range(40)] + [f'ddMFCC_{i}_std' for i in range(40)] +
    [f'Chroma_{i}_mean' for i in range(12)] + [f'Chroma_{i}_std' for i in range(12)] +
    ['SpCentroid_mean', 'SpCentroid_std', 'SpBandwidth_mean', 'SpBandwidth_std',
     'SpRolloff_mean',  'SpRolloff_std',  'SpFlatness_mean',  'SpFlatness_std',
     'ZCR_mean', 'ZCR_std', 'RMS_mean', 'RMS_std', 'Pitch_mean', 'Pitch_std']
)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_k), importances[top_idx][::-1], color='mediumpurple')
ax.set_yticks(range(top_k))
ax.set_yticklabels([feat_names[i] if i < len(feat_names) else f'feat_{i}'
                    for i in top_idx[::-1]], fontsize=8)
ax.set_xlabel('Importance')
ax.set_title(f'Top {top_k} Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_feature_importance.png'), dpi=150)
plt.show()

## 5. CNN (Log-Mel Spectrogram)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Dataset ───────────────────────────────────────────────────────────────
class MelDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# ── Model ─────────────────────────────────────────────────────────────────
class BirdCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

# ── Load data ─────────────────────────────────────────────────────────────
Xc  = np.load(os.path.join(FEATURES_DIR, 'X_mel.npy'))
yc  = np.load(os.path.join(FEATURES_DIR, 'y.npy'))
lnc = np.load(os.path.join(FEATURES_DIR, 'label_names.npy'))
with open(os.path.join(FEATURES_DIR, 'class_info.json')) as f:
    cic = json.load(f)
n_cls = len(lnc)

Xc_tmp, Xc_te, yc_tmp, yc_te = train_test_split(Xc, yc, test_size=TEST_SIZE,
                                                   stratify=yc, random_state=RANDOM_STATE)
val_frac = VAL_SIZE / (1 - TEST_SIZE)
Xc_tr, Xc_val, yc_tr, yc_val = train_test_split(Xc_tmp, yc_tmp, test_size=val_frac,
                                                   stratify=yc_tmp, random_state=RANDOM_STATE)
print(f'Train: {len(yc_tr)}  Val: {len(yc_val)}  Test: {len(yc_te)}')

cc = np.bincount(yc_tr, minlength=n_cls)
ww = 1.0 / cc[yc_tr]
sampler_c = WeightedRandomSampler(ww, len(yc_tr), replacement=True)

train_dl_c = DataLoader(MelDataset(Xc_tr,  yc_tr),  batch_size=BATCH_SIZE, sampler=sampler_c)
val_dl_c   = DataLoader(MelDataset(Xc_val, yc_val), batch_size=BATCH_SIZE, shuffle=False)
test_dl_c  = DataLoader(MelDataset(Xc_te,  yc_te),  batch_size=BATCH_SIZE, shuffle=False)

# ── Training ──────────────────────────────────────────────────────────────
model_c   = BirdCNN(n_cls).to(DEVICE)
crit_c    = nn.CrossEntropyLoss()
opt_c     = optim.AdamW(model_c.parameters(), lr=LR, weight_decay=1e-4)
sched_c   = optim.lr_scheduler.CosineAnnealingLR(opt_c, T_max=EPOCHS_CNN)

def run_epoch_c(loader, training=True):
    model_c.train() if training else model_c.eval()
    tot_loss, correct, n = 0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            if training: opt_c.zero_grad()
            out  = model_c(Xb)
            loss = crit_c(out, yb)
            if training: loss.backward(); opt_c.step()
            tot_loss += loss.item() * len(yb)
            correct  += (out.argmax(1) == yb).sum().item()
            n        += len(yb)
    return tot_loss / n, correct / n

best_val_acc_c, pat_c = 0.0, 0
tr_losses_c, vl_losses_c, tr_accs_c, vl_accs_c = [], [], [], []

print(f'\nTraining CNN for up to {EPOCHS_CNN} epochs…')
for ep in range(1, EPOCHS_CNN + 1):
    tl, ta = run_epoch_c(train_dl_c, True)
    vl, va = run_epoch_c(val_dl_c,   False)
    sched_c.step()
    tr_losses_c.append(tl); vl_losses_c.append(vl)
    tr_accs_c.append(ta);   vl_accs_c.append(va)
    print(f'  Ep {ep:3d}/{EPOCHS_CNN} | Train {tl:.4f}/{ta:.4f} | Val {vl:.4f}/{va:.4f}')
    if va > best_val_acc_c:
        best_val_acc_c = va; pat_c = 0
        torch.save(model_c.state_dict(), os.path.join(MODELS_DIR, 'cnn_best.pt'))
    else:
        pat_c += 1
        if pat_c >= PATIENCE:
            print(f'  Early stopping at epoch {ep}'); break

model_c.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'cnn_best.pt'), map_location=DEVICE))

In [ ]:
# CNN training curves
ep_range = range(1, len(tr_losses_c) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep_range, tr_losses_c, label='Train'); axes[0].plot(ep_range, vl_losses_c, label='Val')
axes[0].set(title='CNN Loss', xlabel='Epoch', ylabel='Loss'); axes[0].legend()
axes[1].plot(ep_range, tr_accs_c,  label='Train'); axes[1].plot(ep_range, vl_accs_c,  label='Val')
axes[1].set(title='CNN Accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cnn_training_curves.png'), dpi=150)
plt.show()

# CNN test evaluation
model_c.eval()
all_preds_c, all_true_c = [], []
with torch.no_grad():
    for Xb, yb in test_dl_c:
        preds = model_c(Xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds_c.extend(preds); all_true_c.extend(yb.numpy())
all_preds_c = np.array(all_preds_c); all_true_c = np.array(all_true_c)
cnn_acc = (all_preds_c == all_true_c).mean()
cnn_f1  = f1_score(all_true_c, all_preds_c, average='macro')
print(f'CNN Test acc: {cnn_acc:.4f}   Macro F1: {cnn_f1:.4f}')
print(classification_report(all_true_c, all_preds_c, target_names=lnc, zero_division=0))

sn_c = [cic.get(sp, sp)[:12] for sp in lnc]
cm_c = confusion_matrix(all_true_c, all_preds_c)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues',
            xticklabels=sn_c, yticklabels=sn_c, ax=ax, linewidths=0.3)
ax.set(xlabel='Predicted', ylabel='True',
       title=f'CNN Confusion Matrix — Acc={cnn_acc:.3f}  F1={cnn_f1:.3f}')
plt.xticks(rotation=45, ha='right', fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cnn_confusion_matrix.png'), dpi=150)
plt.show()

## 6. Bidirectional LSTM (MFCC Sequence)

In [ ]:
class MFCCDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


class BirdLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, n_classes, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        feat_dim = hidden_size * 2
        self.attention  = nn.Linear(feat_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128),     nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        attn_w  = torch.softmax(self.attention(out), dim=1)
        context = (attn_w * out).sum(dim=1)
        return self.classifier(context)


# ── Load data ─────────────────────────────────────────────────────────────
Xl  = np.load(os.path.join(FEATURES_DIR, 'X_mfcc_seq.npy'))
yl  = np.load(os.path.join(FEATURES_DIR, 'y.npy'))
lnl = np.load(os.path.join(FEATURES_DIR, 'label_names.npy'))
with open(os.path.join(FEATURES_DIR, 'class_info.json')) as f:
    cil = json.load(f)
n_cls_l = len(lnl)
print(f'X shape: {Xl.shape}  n_classes: {n_cls_l}')

Xl_tmp, Xl_te, yl_tmp, yl_te = train_test_split(Xl, yl, test_size=TEST_SIZE,
                                                   stratify=yl, random_state=RANDOM_STATE)
Xl_tr, Xl_val, yl_tr, yl_val = train_test_split(Xl_tmp, yl_tmp, test_size=val_frac,
                                                   stratify=yl_tmp, random_state=RANDOM_STATE)
print(f'Train: {len(yl_tr)}  Val: {len(yl_val)}  Test: {len(yl_te)}')

ccl = np.bincount(yl_tr, minlength=n_cls_l)
wwl = 1.0 / ccl[yl_tr]
sampler_l = WeightedRandomSampler(wwl, len(yl_tr), replacement=True)

train_dl_l = DataLoader(MFCCDataset(Xl_tr,  yl_tr),  batch_size=BATCH_SIZE, sampler=sampler_l)
val_dl_l   = DataLoader(MFCCDataset(Xl_val, yl_val), batch_size=BATCH_SIZE, shuffle=False)
test_dl_l  = DataLoader(MFCCDataset(Xl_te,  yl_te),  batch_size=BATCH_SIZE, shuffle=False)

# ── Training ──────────────────────────────────────────────────────────────
model_l  = BirdLSTM(input_size=N_MFCC, hidden_size=128, num_layers=2,
                     n_classes=n_cls_l, dropout=0.3).to(DEVICE)
print(f'BiLSTM parameters: {sum(p.numel() for p in model_l.parameters()):,}')

crit_l  = nn.CrossEntropyLoss()
opt_l   = optim.AdamW(model_l.parameters(), lr=LR, weight_decay=1e-4)
sched_l = optim.lr_scheduler.ReduceLROnPlateau(opt_l, patience=3, factor=0.5)

def run_epoch_l(loader, training=True):
    model_l.train() if training else model_l.eval()
    tot_loss, correct, n = 0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            if training: opt_l.zero_grad()
            out  = model_l(Xb)
            loss = crit_l(out, yb)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model_l.parameters(), 1.0)
                opt_l.step()
            tot_loss += loss.item() * len(yb)
            correct  += (out.argmax(1) == yb).sum().item()
            n        += len(yb)
    return tot_loss / n, correct / n

best_val_acc_l, pat_l = 0.0, 0
tr_losses_l, vl_losses_l, tr_accs_l, vl_accs_l = [], [], [], []

print(f'\nTraining BiLSTM for up to {EPOCHS_LSTM} epochs…')
for ep in range(1, EPOCHS_LSTM + 1):
    tl, ta = run_epoch_l(train_dl_l, True)
    vl, va = run_epoch_l(val_dl_l,   False)
    sched_l.step(vl)
    tr_losses_l.append(tl); vl_losses_l.append(vl)
    tr_accs_l.append(ta);   vl_accs_l.append(va)
    print(f'  Ep {ep:3d}/{EPOCHS_LSTM} | Train {tl:.4f}/{ta:.4f} | Val {vl:.4f}/{va:.4f}')
    if va > best_val_acc_l:
        best_val_acc_l = va; pat_l = 0
        torch.save(model_l.state_dict(), os.path.join(MODELS_DIR, 'lstm_best.pt'))
    else:
        pat_l += 1
        if pat_l >= PATIENCE:
            print(f'  Early stopping at epoch {ep}'); break

model_l.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'lstm_best.pt'), map_location=DEVICE))

In [ ]:
# BiLSTM training curves
ep_range_l = range(1, len(tr_losses_l) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep_range_l, tr_losses_l, label='Train'); axes[0].plot(ep_range_l, vl_losses_l, label='Val')
axes[0].set(title='BiLSTM Loss', xlabel='Epoch', ylabel='Loss'); axes[0].legend()
axes[1].plot(ep_range_l, tr_accs_l,  label='Train'); axes[1].plot(ep_range_l, vl_accs_l,  label='Val')
axes[1].set(title='BiLSTM Accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'lstm_training_curves.png'), dpi=150)
plt.show()

# BiLSTM test evaluation
model_l.eval()
all_preds_l, all_true_l = [], []
with torch.no_grad():
    for Xb, yb in test_dl_l:
        preds = model_l(Xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds_l.extend(preds); all_true_l.extend(yb.numpy())
all_preds_l = np.array(all_preds_l); all_true_l = np.array(all_true_l)
lstm_acc = (all_preds_l == all_true_l).mean()
lstm_f1  = f1_score(all_true_l, all_preds_l, average='macro')
print(f'BiLSTM Test acc: {lstm_acc:.4f}   Macro F1: {lstm_f1:.4f}')
print(classification_report(all_true_l, all_preds_l, target_names=lnl, zero_division=0))

sn_l = [cil.get(sp, sp)[:12] for sp in lnl]
cm_l = confusion_matrix(all_true_l, all_preds_l)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm_l, annot=True, fmt='d', cmap='Greens',
            xticklabels=sn_l, yticklabels=sn_l, ax=ax, linewidths=0.3)
ax.set(xlabel='Predicted', ylabel='True',
       title=f'BiLSTM Confusion Matrix — Acc={lstm_acc:.3f}  F1={lstm_f1:.3f}')
plt.xticks(rotation=45, ha='right', fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'lstm_confusion_matrix.png'), dpi=150)
plt.show()

## 7. 最终结果汇总

In [ ]:
summary = pd.DataFrame([
    {'Model': 'KNN',          'Test Accuracy': r_knn['test_acc'], 'Macro F1': r_knn['macro_f1']},
    {'Model': 'SVM',          'Test Accuracy': r_svm['test_acc'], 'Macro F1': r_svm['macro_f1']},
    {'Model': 'Random Forest', 'Test Accuracy': r_rf['test_acc'],  'Macro F1': r_rf['macro_f1']},
    {'Model': 'CNN',          'Test Accuracy': cnn_acc,           'Macro F1': cnn_f1},
    {'Model': 'BiLSTM',       'Test Accuracy': lstm_acc,          'Macro F1': lstm_f1},
]).set_index('Model')

print('\n=== Final Results ===')
print(summary.to_string(float_format='{:.4f}'.format))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(summary)); w = 0.35
ax.bar(x - w/2, summary['Test Accuracy'], w, label='Test Accuracy', color='steelblue')
ax.bar(x + w/2, summary['Macro F1'],      w, label='Macro F1',      color='coral')
ax.set_xticks(x); ax.set_xticklabels(summary.index, rotation=15)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('All Models — Test Accuracy vs Macro F1'); ax.legend()
for i, (acc, f1) in enumerate(zip(summary['Test Accuracy'], summary['Macro F1'])):
    ax.text(i - w/2, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=8)
    ax.text(i + w/2, f1  + 0.01, f'{f1:.3f}',  ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'final_comparison.png'), dpi=150)
plt.show()
summary.to_csv(os.path.join(RESULTS_DIR, 'final_results.csv'))

print(f'\nAll outputs saved to: {RESULTS_DIR}')